In [ ]:
%%writefile requirements.txt
optuna==5.0.0
scikit-learn==1.5.0
scipy==1.13.1
seaborn==0.13.2
matplotlib==3.9.0
lime==0.2.0.1
shap==0.45.0
opencv-python-headless==4.10.0.84
pandas==2.2.2

# Random Forest Convolucional (algoritmo da IC) — Setup e Importações

Este notebook reproduz, para o **Random Forest**, o mesmo protocolo experimental já aplicado
às CNNs (`mobilenet.ipynb`) e ao ViT (`transformers.ipynb`): mesmos datasets e splits, busca de
hiperparâmetros com Optuna (F1-Macro na validação), treino definitivo, métricas no teste,
curvas ROC/PR, calibração, predição seletiva, XAI, latência, exportação, validação multi-seed,
tabela comparativa, teste de significância e automação da **Avaliação Cruzada (Modelo × Dataset)**.

O extrator de características é o mesmo da IC ("Convolutional Random Forest"): imagem em tons
de cinza → filtros Gaussiano, Sobel, Laplaciano, Mediana e Max Pooling → média e desvio-padrão
de cada mapa (10 características por imagem).

Diferenças em relação ao notebook da IC (correções, não mudanças de método):
- A IC usava `next(iter(train_dataset))` com `batch_size=2000`, ou seja, treinava só com as
  **primeiras 2000 imagens** do treino (e 500 do teste). Aqui são usadas **todas** as imagens.
- A IC não usava o conjunto de validação. Aqui ele é usado para a busca do Optuna e para a
  calibração de confiança (o teste continua intocado até a avaliação final).
- O Max Pooling foi vetorizado (`reshape` + `max`) — resultado numericamente idêntico ao laço
  duplo original, só que ordens de grandeza mais rápido.

In [ ]:
# ── Instala dependências com versões fixadas (mesma estratégia do mobilenet.ipynb) ──────
# Ver o comentário completo no mobilenet.ipynb: --force-reinstall --ignore-installed
# --no-deps --no-cache-dir evita instalações "misturadas" de versões do mesmo pacote na
# imagem base do Kaggle.
!pip install -q --force-reinstall --ignore-installed --no-deps --no-cache-dir -r requirements.txt

# ── Exportação ONNX (opcional) ───────────────────────────────────────────────
# ONNX é o formato "exportado" do Random Forest (equivalente ao .tflite das CNNs e ao .ptl do
# ViT): roda no Flutter via onnxruntime, sem Python. Instalado separado e SEM --no-deps
# forçado de propósito -- se falhar, o notebook segue normalmente e a seção de exportação cai
# para o formato joblib (sklearn), marcando isso na tabela comparativa. "protobuf<5": com
# protobuf >= 5 o skl2onnx falha ao montar o nó TreeEnsembleClassifier ("Expected an int, got a
# boolean"); o TensorFlow do Kaggle já exige protobuf < 5, então isso não troca a versão dele.
!pip install -q --no-cache-dir skl2onnx==1.17.0 onnx==1.16.1 onnxruntime==1.18.1 "protobuf<5" || echo "Aviso: ONNX indisponível -- exportação cairá para joblib."

import os
import gc
import time
import json
import glob
import joblib
import pathlib
import pandas as pd
import numpy as np
import optuna
import seaborn as sns
import matplotlib.pyplot as plt

from collections import Counter
from joblib import Parallel, delayed
from scipy import ndimage
from scipy.signal import convolve2d
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)

# TensorFlow é usado SÓ para ler/redimensionar as imagens (image_dataset_from_directory),
# garantindo exatamente o mesmo pré-processamento de pixels (resize bilinear 224x224) usado
# pelas CNNs e pela IC. O Random Forest roda 100% em CPU -- esconder a GPU evita que o TF
# reserve VRAM à toa.
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')

import sklearn
print(f"scikit-learn: {sklearn.__version__} | Optuna: {optuna.__version__} | TensorFlow: {tf.__version__}")

# ── Reprodutibilidade ──────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
print(f"Seeds fixados: SEED={SEED}")

# ── Configuração de Datasets (IDÊNTICA ao mobilenet.ipynb) ──────────────────────
# Mesmos caminhos e mesmos splits -- condição para a Avaliação Cruzada e o teste de
# significância compararem RF × CNN × ViT sobre exatamente as mesmas imagens de teste.
DATASETS = {
    'bracol': {
        'train':     '/kaggle/input/newdataset/new_dataset/train',
        'val':       '/kaggle/input/newdataset/new_dataset/val',
        'test':      '/kaggle/input/newdataset/new_dataset/test',
        'n_classes': 5,   # Ferrugem, Bicho-Mineiro, Cercosporiose, Phoma e Saudável (Tabela 1 da proposta)
    },
    'rocole': {
        'train':     '/kaggle/working/datasets/rocole/train',
        'val':       '/kaggle/working/datasets/rocole/val',
        'test':      '/kaggle/working/datasets/rocole/test',
        'n_classes': 3,   # Ferrugem, Ácaro Vermelho e Saudável (Tabela 1 da proposta)
    },
    'coffee_leaf_diseases': {
        'train':     '/kaggle/working/datasets/coffee_leaf_diseases/train',
        'val':       '/kaggle/working/datasets/coffee_leaf_diseases/val',
        'test':      '/kaggle/working/datasets/coffee_leaf_diseases/test',
        'n_classes': 4,   # Bicho-Mineiro, Ferrugem, Phoma e Saudável (sem Cercosporiose -- não existe neste dataset)
    },
    'jmuben': {
        'train':     '/kaggle/working/datasets/jmuben/train',
        'val':       '/kaggle/working/datasets/jmuben/val',
        'test':      '/kaggle/working/datasets/jmuben/test',
        'n_classes': 5,   # Ferrugem, Bicho-Mineiro, Cercosporiose, Phoma e Saudável (Tabela 1 da proposta)
    },
}

DATASET_ATIVO = 'bracol'

# Nome usado na coluna "Modelo" da tabela comparativa e nos nomes de pasta/arquivo --
# outputs/RandomForest_{dataset}/, no mesmo esquema {modelo}_{dataset} das CNNs e do ViT.
MODELO_NOME = 'RandomForest'

img_size   = (224, 224)
batch_size = 32


def ativar_dataset(dataset_name):
    """Atualiza as globais derivadas do dataset ativo (mesmo papel do bloco "Derivados da
    configuração ativa" do mobilenet.ipynb), para que as células manuais e a automação da
    Avaliação Cruzada compartilhem o mesmo estado."""
    global DATASET_ATIVO, N_CLASSES, NOME_ARQUIVO, OUTPUT_DIR
    DATASET_ATIVO = dataset_name
    N_CLASSES     = DATASETS[DATASET_ATIVO]['n_classes']
    NOME_ARQUIVO  = f"{MODELO_NOME}_{DATASET_ATIVO}"
    OUTPUT_DIR    = os.path.join('outputs', NOME_ARQUIVO)
    os.makedirs(OUTPUT_DIR, exist_ok=True)


ativar_dataset(DATASET_ATIVO)
print(f"Dataset ativo : {DATASET_ATIVO} ({N_CLASSES} classes)")
print(f"Modelo        : {MODELO_NOME}")
print(f"Diretório de saída: {OUTPUT_DIR}")

# Preparação dos Datasets (RoCoLe, coffee-leaf-diseases e JMuBEN — divisão Train/Val/Test)

Cópia literal da célula de preparação do `mobilenet.ipynb` (mesmas sementes, mesmas
proporções, mesmos nomes de pasta "train"/"val"/"test"). Se o split já existir em
`/kaggle/working/datasets/` (ex.: gerado pelo notebook das CNNs na mesma sessão), é reaproveitado.

In [ ]:
# O RoCoLe (diferente do newdataset e do BRACOL) não vem pré-dividido em train/val/test --
# precisa ser separado antes do image_dataset_from_directory conseguir ler os splits. Usa
# os nomes de pasta "train"/"val"/"test" (convenção deste notebook Keras) -- no
# transformers.ipynb a mesma preparação grava "validation" em vez de "val", por causa do
# load_dataset("imagefolder") do Hugging Face, que não reconhece "val" como nome de split.
def _split_ja_populado(split_dir):
    """True se split_dir existir e "train/" dentro dela tiver pelo menos um arquivo.
    Só checar split_dir.exists() não basta: se uma tentativa anterior criou as pastas de
    classe mas copiou 0 imagens (raw_dir errado, CSV com id que não bate, etc.), a
    próxima execução via idempotência pulava para sempre achando que já estava pronto."""
    train_dir = split_dir / "train"
    return split_dir.exists() and train_dir.exists() and any(
        p.is_file() for p in train_dir.rglob("*")
    )


def _localizar_raw_dir_rocole():
    """O caminho exato do RoCoLe dentro de /kaggle/input varia conforme como o dataset foi
    anexado -- às vezes os arquivos ficam direto na raiz (/kaggle/input/rocole/), às vezes
    dentro de uma subpasta extra com o nome do dataset (ex.: RoCoLe-dataset). Em vez de
    fixar um único palpite, tenta os candidatos mais prováveis e usa o primeiro que existir
    e contiver pelo menos 2 subpastas (as classes)."""
    import pathlib
    candidatos = [
        pathlib.Path("/kaggle/input/rocole/RoCoLe-dataset"),
        pathlib.Path("/kaggle/input/datasets/nirmalsankalana/rocole-a-robusta-coffee-leaf-images-dataset"),
        pathlib.Path("/kaggle/input/rocole"),
    ]
    for cand in candidatos:
        if cand.exists() and sum(1 for d in cand.iterdir() if d.is_dir()) >= 2:
            return cand
    raise FileNotFoundError(
        f"Não encontrei a pasta bruta do RoCoLe em nenhum destes caminhos: {candidatos}. "
        "Rode `!find /kaggle/input/rocole -maxdepth 2` numa célula para achar o caminho "
        "real e ajuste ROCOLE_RAW_DIR manualmente."
    )


def preparar_rocole():
    """Divide o RoCoLe bruto em train/val/test (70/15/15) se ainda não foi feito.
    Idempotente: se a pasta de destino já existir, não refaz o split."""
    import pathlib
    import shutil
    from sklearn.model_selection import train_test_split

    raw_dir   = _localizar_raw_dir_rocole()
    split_dir = pathlib.Path(DATASETS['rocole']['train']).parent

    if _split_ja_populado(split_dir):
        print(f"Split do RoCoLe já existe em {split_dir} -- pulando (apague a pasta se quiser regerar).")
        return

    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
    classes_rocole = sorted([d.name for d in raw_dir.iterdir() if d.is_dir()])
    print(f"Classes encontradas no RoCoLe: {classes_rocole}")

    for split in ["train", "val", "test"]:
        for classe in classes_rocole:
            (split_dir / split / classe).mkdir(parents=True, exist_ok=True)

    for classe in classes_rocole:
        arquivos = sorted([
            p for p in (raw_dir / classe).iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ])

        treino, temp = train_test_split(arquivos, train_size=TRAIN_RATIO, random_state=SEED, shuffle=True)
        val_frac_do_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
        val, teste = train_test_split(temp, train_size=val_frac_do_temp, random_state=SEED, shuffle=True)

        for grupo, nome_split in [(treino, "train"), (val, "val"), (teste, "test")]:
            for arquivo in grupo:
                shutil.copy2(arquivo, split_dir / nome_split / classe / arquivo.name)

        print(f"  [{classe}] treino={len(treino)} val={len(val)} teste={len(teste)}")

    print(f"RoCoLe dividido salvo em: {split_dir}")


# ── Preparação do coffee-leaf-diseases (badasstechie) ──────────────────────────────
# Dataset original é multi-rótulo por imagem (colunas binárias independentes
# miner/rust/phoma no CSV, mais máscaras de segmentação por pixel) -- incompatível com o
# esquema pasta-por-classe do image_dataset_from_directory. Decisão de projeto: descartar
# as imagens com mais de uma doença marcada (~4% do treino, ~3% do teste) e tratar o
# restante como classificação single-label de 4 classes (Bicho-Mineiro, Ferrugem, Phoma e
# Saudável -- sem Cercosporiose, que não existe neste dataset). Usa o teste oficial do
# Kaggle sem alterações e reserva 15% do treino oficial para validação (mesma proporção
# do split do RoCoLe), já que o dataset não vem com uma pasta "val" própria.
_COFFEE_LEAF_COLUNAS = ['miner', 'rust', 'phoma']
_COFFEE_LEAF_CLASSES = {'miner': 'bicho_mineiro', 'rust': 'ferrugem', 'phoma': 'phoma'}
_COFFEE_LEAF_SAUDAVEL = 'saudavel'


def _localizar_raw_dir_coffee_leaf_diseases():
    """Mesma ideia do RoCoLe: o caminho exato varia conforme como o dataset foi anexado
    no Kaggle. Tenta os candidatos mais prováveis e usa o primeiro que contiver os CSVs
    de rótulo (train_classes.csv / test_classes.csv) -- confirmado via
    `find /kaggle/input/datasets/badasstechie/coffee-leaf-diseases -maxdepth 3` que eles
    ficam direto nesta pasta; as imagens em si ficam um nível mais fundo (ver
    _imagens_dir_coffee_leaf logo abaixo)."""
    import pathlib
    candidatos = [
        pathlib.Path("/kaggle/input/datasets/badasstechie/coffee-leaf-diseases"),
        pathlib.Path("/kaggle/input/coffee-leaf-diseases"),
    ]
    for cand in candidatos:
        if (cand / "train_classes.csv").exists() and (cand / "test_classes.csv").exists():
            return cand
    raise FileNotFoundError(
        f"Não encontrei a pasta bruta do coffee-leaf-diseases em nenhum destes caminhos: "
        f"{candidatos}. Rode `!find /kaggle/input -maxdepth 4 -iname '*coffee*'` numa "
        "célula para achar o caminho real e ajuste a lista de candidatos."
    )


def _imagens_dir_coffee_leaf(raw_dir):
    """As imagens (train/images, test/images) NÃO ficam direto em raw_dir -- ficam dentro
    de uma subpasta com o MESMO NOME do dataset, um nível abaixo de onde estão os CSVs
    (confirmado via find: o zip original tinha uma pasta de topo chamada
    "coffee-leaf-diseases" e o Kaggle preservou essa estrutura ao montar em
    /kaggle/input). Checa os dois formatos -- se o Kaggle algum dia remontar sem esse
    nível extra, continua funcionando."""
    direto = raw_dir / "train" / "images"
    aninhado = raw_dir / raw_dir.name / "train" / "images"
    if direto.exists():
        return raw_dir
    if aninhado.exists():
        return raw_dir / raw_dir.name
    raise FileNotFoundError(
        f"Não encontrei train/images nem em {direto} nem em {aninhado}. Rode "
        f"`!find {raw_dir} -maxdepth 3` numa célula para conferir a estrutura real."
    )


def _rotulo_unico_coffee_leaf(linha):
    """Converte uma linha do CSV (miner/rust/phoma binários) num nome de classe único.
    Retorna None quando mais de uma doença está marcada (imagem descartada)."""
    ativas = [col for col in _COFFEE_LEAF_COLUNAS if int(linha[col]) == 1]
    if len(ativas) > 1:
        return None
    if not ativas:
        return _COFFEE_LEAF_SAUDAVEL
    return _COFFEE_LEAF_CLASSES[ativas[0]]


def preparar_coffee_leaf_diseases():
    """Reorganiza o coffee-leaf-diseases bruto (imagens + CSV multi-rótulo) em
    train/val/test de pasta-por-classe. Idempotente: se a pasta de destino já existir,
    não refaz o split."""
    import pathlib
    import shutil
    import csv
    from sklearn.model_selection import train_test_split

    raw_dir   = _localizar_raw_dir_coffee_leaf_diseases()
    imagens_dir = _imagens_dir_coffee_leaf(raw_dir)
    split_dir = pathlib.Path(DATASETS['coffee_leaf_diseases']['train']).parent

    if _split_ja_populado(split_dir):
        print(f"Split do coffee-leaf-diseases já existe em {split_dir} -- pulando (apague a pasta se quiser regerar).")
        return

    classes_ativas = sorted(set(_COFFEE_LEAF_CLASSES.values()) | {_COFFEE_LEAF_SAUDAVEL})
    for split in ["train", "val", "test"]:
        for classe in classes_ativas:
            (split_dir / split / classe).mkdir(parents=True, exist_ok=True)

    def _carregar_rotulos(csv_path):
        rotulos = {}
        descartadas = 0
        with open(csv_path, newline='') as f:
            for linha in csv.DictReader(f):
                classe = _rotulo_unico_coffee_leaf(linha)
                if classe is None:
                    descartadas += 1
                    continue
                rotulos[linha['id']] = classe
        return rotulos, descartadas

    rotulos_treino, descartadas_treino = _carregar_rotulos(raw_dir / "train_classes.csv")
    rotulos_teste,  descartadas_teste  = _carregar_rotulos(raw_dir / "test_classes.csv")
    print(f"Treino: {len(rotulos_treino)} imagens rotuladas, {descartadas_treino} descartadas (multi-rótulo).")
    print(f"Teste : {len(rotulos_teste)} imagens rotuladas, {descartadas_teste} descartadas (multi-rótulo).")

    # Teste oficial do Kaggle vai direto para test/, sem split adicional.
    for id_img, classe in rotulos_teste.items():
        origem = imagens_dir / "test" / "images" / f"{id_img}.jpg"
        shutil.copy2(origem, split_dir / "test" / classe / origem.name)

    # Treino oficial é dividido 85/15 em train/val (estratificado por classe).
    ids = list(rotulos_treino.keys())
    classes_por_id = [rotulos_treino[i] for i in ids]
    ids_treino, ids_val = train_test_split(
        ids, test_size=0.15, random_state=SEED, shuffle=True, stratify=classes_por_id
    )

    for grupo, nome_split in [(ids_treino, "train"), (ids_val, "val")]:
        for id_img in grupo:
            classe = rotulos_treino[id_img]
            origem = imagens_dir / "train" / "images" / f"{id_img}.jpg"
            shutil.copy2(origem, split_dir / nome_split / classe / origem.name)

    for classe in classes_ativas:
        n_tr  = sum(1 for i in ids_treino if rotulos_treino[i] == classe)
        n_val = sum(1 for i in ids_val if rotulos_treino[i] == classe)
        n_te  = sum(1 for c in rotulos_teste.values() if c == classe)
        print(f"  [{classe}] treino={n_tr} val={n_val} teste={n_te}")

    print(f"coffee-leaf-diseases dividido salvo em: {split_dir}")


# ── Preparação do JMuBEN ────────────────────────────────────────────────────────────
# Vem em pasta-por-classe (Cerscospora/Healthy/Leaf rust/Miner/Phoma), igual ao RoCoLe
# bruto -- só que os nomes de pasta usam a nomenclatura em inglês do dataset original, e
# não o vocabulário em português (ferrugem/bicho_mineiro/...) já usado nas classes do
# BRACOL/coffee-leaf-diseases. _JMUBEN_CLASSES normaliza isso para as 5 combinações
# ficarem com os MESMOS nomes de classe do BRACOL, o que facilita comparar matriz de
# confusão / Grad-CAM entre os dois datasets na Avaliação Cruzada.
_JMUBEN_CLASSES = {
    'cerscospora': 'cercosporiose',
    'healthy':     'saudavel',
    'leaf rust':   'ferrugem',
    'miner':       'bicho_mineiro',
    'phoma':       'phoma',
}


def _localizar_raw_dir_jmuben():
    """O caminho exato varia conforme como o dataset foi anexado no Kaggle -- tenta os
    candidatos mais prováveis e usa o primeiro que existir e contiver pelo menos 2
    subpastas (as classes)."""
    import pathlib
    candidatos = [
        pathlib.Path("/kaggle/input/datasets/noamaanabdulazeem/jmuben-coffee-dataset/JMuBEN"),
        pathlib.Path("/kaggle/input/jmuben-coffee-dataset/JMuBEN"),
        pathlib.Path("/kaggle/input/jmuben-coffee-dataset"),
    ]
    for cand in candidatos:
        if cand.exists() and sum(1 for d in cand.iterdir() if d.is_dir()) >= 2:
            return cand
    raise FileNotFoundError(
        f"Não encontrei a pasta bruta do JMuBEN em nenhum destes caminhos: {candidatos}. "
        "Rode `!find /kaggle/input -maxdepth 4 -iname '*jmuben*'` numa célula para achar "
        "o caminho real e ajuste a lista de candidatos."
    )


def preparar_jmuben():
    """Divide o JMuBEN bruto em train/val/test (70/15/15) se ainda não foi feito, e
    normaliza os nomes de pasta para o vocabulário de classes do BRACOL. Idempotente: se
    a pasta de destino já existir, não refaz o split."""
    import pathlib
    import shutil
    from sklearn.model_selection import train_test_split

    raw_dir   = _localizar_raw_dir_jmuben()
    split_dir = pathlib.Path(DATASETS['jmuben']['train']).parent

    if _split_ja_populado(split_dir):
        print(f"Split do JMuBEN já existe em {split_dir} -- pulando (apague a pasta se quiser regerar).")
        return

    TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15
    pastas_brutas = sorted([d.name for d in raw_dir.iterdir() if d.is_dir()])

    nao_mapeadas = [p for p in pastas_brutas if p.strip().lower() not in _JMUBEN_CLASSES]
    if nao_mapeadas:
        raise KeyError(
            f"Pasta(s) do JMuBEN sem mapeamento em _JMUBEN_CLASSES: {nao_mapeadas}. "
            f"Pastas encontradas: {pastas_brutas}. Ajuste _JMUBEN_CLASSES antes de continuar."
        )

    print(f"Classes encontradas no JMuBEN: {pastas_brutas}")

    for split in ["train", "val", "test"]:
        for classe in set(_JMUBEN_CLASSES.values()):
            (split_dir / split / classe).mkdir(parents=True, exist_ok=True)

    for pasta_bruta in pastas_brutas:
        classe = _JMUBEN_CLASSES[pasta_bruta.strip().lower()]
        arquivos = sorted([
            p for p in (raw_dir / pasta_bruta).iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ])

        treino, temp = train_test_split(arquivos, train_size=TRAIN_RATIO, random_state=SEED, shuffle=True)
        val_frac_do_temp = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
        val, teste = train_test_split(temp, train_size=val_frac_do_temp, random_state=SEED, shuffle=True)

        for grupo, nome_split in [(treino, "train"), (val, "val"), (teste, "test")]:
            for arquivo in grupo:
                shutil.copy2(arquivo, split_dir / nome_split / classe / arquivo.name)

        print(f"  [{pasta_bruta} -> {classe}] treino={len(treino)} val={len(val)} teste={len(teste)}")

    print(f"JMuBEN dividido salvo em: {split_dir}")


if DATASET_ATIVO == 'rocole':
    preparar_rocole()
elif DATASET_ATIVO == 'coffee_leaf_diseases':
    preparar_coffee_leaf_diseases()
elif DATASET_ATIVO == 'jmuben':
    preparar_jmuben()
else:
    print(f"Dataset ativo é '{DATASET_ATIVO}' -- nenhuma preparação de split necessária (só RoCoLe, coffee-leaf-diseases e JMuBEN precisam).")

# Extração de Características (algoritmo da IC)

In [ ]:
# ── Filtros convolucionais da IC (mesmas operações, mesmos parâmetros) ────────────────
def apply_convolution(image, kernel):
    return convolve2d(image, kernel, mode='same', boundary='symm')

def apply_gaussian(image):
    return ndimage.gaussian_filter(image.astype(np.float32), sigma=1)

def apply_sobel(image):
    kernel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
    kernel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])
    grad_x = apply_convolution(image, kernel_x)
    grad_y = apply_convolution(image, kernel_y)
    return np.hypot(grad_x, grad_y)

def apply_laplacian(image):
    return ndimage.laplace(image.astype(np.float32))

def apply_median(image):
    # astype(np.uint8) trunca a média RGB (float) para inteiro -- comportamento original da
    # IC, mantido de propósito para não alterar o extrator.
    return ndimage.median_filter(image.astype(np.uint8), size=3)

def apply_max_pooling(image, pool_size=2):
    # Versão vetorizada do laço duplo da IC: mesmas janelas pool_size x pool_size sem
    # sobreposição, mesmo descarte da borda quando a dimensão não é múltipla de pool_size.
    h = image.shape[0] // pool_size
    w = image.shape[1] // pool_size
    recorte = image[:h * pool_size, :w * pool_size]
    return recorte.reshape(h, pool_size, w, pool_size).max(axis=(1, 3))

def extract_features(image):
    gray_image = np.mean(image, axis=-1)

    blurred    = apply_gaussian(gray_image)
    sobel      = apply_sobel(gray_image)
    laplacian  = apply_laplacian(gray_image)
    median     = apply_median(gray_image)
    max_pooled = apply_max_pooling(blurred)

    return [
        np.mean(blurred),    np.std(blurred),
        np.mean(sobel),      np.std(sobel),
        np.mean(laplacian),  np.std(laplacian),
        np.mean(median),     np.std(median),
        np.mean(max_pooled), np.std(max_pooled),
    ]

FEATURE_NAMES = [
    'Média (Gaussiano)',   'Desvio Padrão (Gaussiano)',
    'Média (Sobel)',       'Desvio Padrão (Sobel)',
    'Média (Laplaciano)',  'Desvio Padrão (Laplaciano)',
    'Média (Mediana)',     'Desvio Padrão (Mediana)',
    'Média (Max Pooling)', 'Desvio Padrão (Max Pooling)',
]


def extrair_features_lote(imagens, n_jobs=-1):
    """Extrai as características de um lote de imagens em paralelo (todos os núcleos de
    CPU). Lotes pequenos rodam em série -- o overhead de despachar processos não compensa."""
    if len(imagens) < 16:
        return np.array([extract_features(img) for img in imagens], dtype=np.float32)
    feats = Parallel(n_jobs=n_jobs)(delayed(extract_features)(img) for img in imagens)
    return np.array(feats, dtype=np.float32)


def carregar_imagem(caminho, target_size=img_size):
    """Lê UMA imagem do disco com o MESMO pipeline do image_dataset_from_directory (decode
    + resize bilinear do TF), devolvendo float32 (H, W, 3) em 0-255 -- é o formato que
    extract_features espera. Usado no LIME, na latência e nas visualizações."""
    raw = tf.io.read_file(str(caminho))
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, target_size, method='bilinear')
    return img.numpy().astype(np.float32)


# Versão do extrator gravada no nome do cache -- mude se alterar extract_features, para não
# reaproveitar características antigas por engano.
FEATURES_VERSAO = 'ic_v1'
FEATURES_CACHE_DIR = os.path.join('outputs', 'features_rf')
os.makedirs(FEATURES_CACHE_DIR, exist_ok=True)


def extrair_split(dataset_name, split):
    """Extrai (X, y, file_paths, class_names) de UM split, com cache em disco (.npz).

    Extrair as features do JMuBEN (~58 mil imagens) leva alguns minutos -- o cache evita
    repetir isso a cada seed, a cada trial do Optuna e entre sessões do Kaggle (basta copiar
    outputs/features_rf/ junto com o resto da pasta outputs/)."""
    cache_path = os.path.join(FEATURES_CACHE_DIR, f'{dataset_name}_{split}_{FEATURES_VERSAO}.npz')
    if os.path.exists(cache_path):
        dados = np.load(cache_path, allow_pickle=True)
        print(f"  [{split}] features carregadas do cache: {cache_path} ({len(dados['y'])} imagens)")
        return dados['X'], dados['y'], list(dados['file_paths']), list(dados['class_names'])

    # shuffle=False: a ordem das features fica alinhada com file_paths (necessário para o
    # LIME/visualizações saberem qual arquivo corresponde a cada predição). A ordem não
    # importa para o Random Forest em si.
    ds = tf.keras.utils.image_dataset_from_directory(
        DATASETS[dataset_name][split], image_size=img_size, batch_size=256,
        label_mode='int', shuffle=False,
    )
    file_paths, class_names = ds.file_paths, ds.class_names

    t0 = time.time()
    X_partes, y_partes = [], []
    for imgs, labels in ds:
        X_partes.append(extrair_features_lote(imgs.numpy()))
        y_partes.append(labels.numpy())
    X = np.concatenate(X_partes, axis=0)
    y = np.concatenate(y_partes, axis=0)
    print(f"  [{split}] {len(y)} imagens -> {X.shape[1]} features em {time.time() - t0:.1f}s")

    np.savez_compressed(cache_path, X=X, y=y, file_paths=np.array(file_paths), class_names=np.array(class_names))
    return X, y, file_paths, class_names


def carregar_features(dataset_name):
    """Prepara o split (se necessário) e carrega as features de train/val/test do dataset."""
    if dataset_name == 'rocole':
        preparar_rocole()
    elif dataset_name == 'coffee_leaf_diseases':
        preparar_coffee_leaf_diseases()
    elif dataset_name == 'jmuben':
        preparar_jmuben()

    print(f"Extraindo/carregando características de '{dataset_name}'...")
    X_tr, y_tr, _,          classes    = extrair_split(dataset_name, 'train')
    X_va, y_va, _,          classes_va = extrair_split(dataset_name, 'val')
    X_te, y_te, paths_test, classes_te = extrair_split(dataset_name, 'test')
    assert classes == classes_va == classes_te, f"Classes divergentes entre splits: {classes} / {classes_va} / {classes_te}"
    return X_tr, y_tr, X_va, y_va, X_te, y_te, paths_test, classes

# Carregamento de Dados

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test, test_file_paths, class_names_geral = carregar_features(DATASET_ATIVO)

print(f"\nClasses: {class_names_geral}")
print(f"Treino: {X_train.shape} | Validação: {X_val.shape} | Teste: {X_test.shape}")
print(f"Distribuição (treino): {dict(sorted(Counter(y_train.tolist()).items()))}")

# Visualização dos Filtros (o que o Random Forest "enxerga")

Equivalente visual às camadas convolucionais das CNNs: cada coluna é um dos mapas cujas
estatísticas (média e desvio-padrão) viram as 10 características do Random Forest.

In [ ]:
fig, axes = plt.subplots(len(class_names_geral), 6, figsize=(18, 3.2 * len(class_names_geral)))
axes = np.atleast_2d(axes)
titulos = ['Original', 'Gaussiano', 'Sobel', 'Laplaciano', 'Mediana', 'Max Pooling']

for class_idx, class_name in enumerate(class_names_geral):
    idx = int(np.where(y_test == class_idx)[0][0])
    img = carregar_imagem(test_file_paths[idx])
    gray = np.mean(img, axis=-1)
    mapas = [
        img.astype(np.uint8), apply_gaussian(gray), apply_sobel(gray),
        apply_laplacian(gray), apply_median(gray), apply_max_pooling(apply_gaussian(gray)),
    ]
    for col, (mapa, titulo) in enumerate(zip(mapas, titulos)):
        ax = axes[class_idx, col]
        ax.imshow(mapa, cmap=None if col == 0 else 'gray')
        ax.set_title(f'{titulo}' + (f'\n{class_name}' if col == 0 else ''), fontsize=9)
        ax.axis('off')

plt.suptitle(f'Mapas de filtros — {MODELO_NOME}/{DATASET_ATIVO}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'filtros_{NOME_ARQUIVO}.png'), dpi=110, bbox_inches='tight')
plt.show()
plt.close()

# Construtor do Modelo

In [ ]:
def criar_rf(trial=None, best_params=None, seed=SEED, n_jobs=-1):
    """Cria o RandomForestClassifier a partir de um trial do Optuna (busca) ou dos
    best_params (treino definitivo / multi-seed) -- mesma ideia do create_model() das CNNs.

    Espaço de busca: o n_estimators=100 da IC fica dentro do intervalo, e class_weight cobre
    o balanceamento por classe que as CNNs fazem via class_weight no model.fit()."""
    if trial:
        params = {
            'n_estimators':      trial.suggest_int('n_estimators', 100, 800, step=50),
            'max_depth':         trial.suggest_categorical('max_depth', [None, 8, 12, 16, 24, 32]),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
            'max_features':      trial.suggest_float('max_features', 0.2, 1.0),
            'criterion':         trial.suggest_categorical('criterion', ['gini', 'entropy']),
            'bootstrap':         trial.suggest_categorical('bootstrap', [True, False]),
            'class_weight':      trial.suggest_categorical('class_weight', [None, 'balanced', 'balanced_subsample']),
        }
    else:
        params = dict(best_params)

    return RandomForestClassifier(**params, random_state=seed, n_jobs=n_jobs)

# Busca do Optuna

Diferente das CNNs, não há necessidade de isolar os trials em subprocessos: o Random Forest
não acumula caches nativos de GPU entre trials. O estudo é persistido em SQLite (retomável
se a sessão cair), com a mesma métrica-alvo das CNNs e do ViT: **F1-Macro na validação**.

In [ ]:
N_TRIALS_RF = 60


def buscar_hiperparametros(n_trials=N_TRIALS_RF):
    """Roda (ou retoma) a busca Optuna para o dataset ativo e persiste study + best_params."""
    study_name = NOME_ARQUIVO
    storage = f"sqlite:///{os.path.join(OUTPUT_DIR, f'optuna_{study_name}.db')}"

    def objective(trial):
        clf = criar_rf(trial=trial)
        clf.fit(X_train, y_train)
        return f1_score(y_val, clf.predict(X_val), average='macro')

    study = optuna.create_study(
        study_name=study_name, storage=storage, direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED), load_if_exists=True,
    )
    restantes = n_trials - len(study.trials)
    if restantes > 0:
        print(f"{NOME_ARQUIVO}: rodando {restantes} trial(s) (estudo já tem {len(study.trials)})...")
        study.optimize(objective, n_trials=restantes, gc_after_trial=True)
    else:
        print(f"{NOME_ARQUIVO}: estudo já tem {len(study.trials)} trial(s) -- nada a rodar.")

    joblib.dump(study, os.path.join(OUTPUT_DIR, f'optuna_study_{NOME_ARQUIVO}.pkl'))
    with open(os.path.join(OUTPUT_DIR, f'best_params_{NOME_ARQUIVO}.json'), 'w') as f:
        json.dump(study.best_params, f, indent=2)
    return study, study.best_params


optuna.logging.set_verbosity(optuna.logging.WARNING)
study, best_params = buscar_hiperparametros(N_TRIALS_RF)
print(f"Melhor F1-Macro (validação): {study.best_value:.4f}")
print("Melhores parâmetros:", json.dumps(best_params, indent=2))

In [ ]:
# ── Histórico e importância dos hiperparâmetros da busca ─────────────────────
df_trials = study.trials_dataframe()
df_trials.to_csv(os.path.join(OUTPUT_DIR, f'optuna_trials_{NOME_ARQUIVO}.csv'), index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
valores = df_trials['value'].values
axes[0].plot(valores, 'o', alpha=0.5, label='Trial')
axes[0].plot(np.maximum.accumulate(np.nan_to_num(valores, nan=0)), 'r-', linewidth=2, label='Melhor até agora')
axes[0].set_title('Histórico da Otimização (F1-Macro na validação)', fontsize=13)
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('F1-Macro')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

try:
    importancias_hp = optuna.importance.get_param_importances(study)
    axes[1].barh(list(importancias_hp.keys())[::-1], list(importancias_hp.values())[::-1], color='steelblue')
    axes[1].set_title('Importância dos Hiperparâmetros (fANOVA)', fontsize=13)
    axes[1].grid(True, alpha=0.3, axis='x')
except Exception as e:  # precisa de >1 trial completo com variação nos parâmetros
    axes[1].text(0.5, 0.5, f'Indisponível: {e}', ha='center', va='center', wrap=True)
    axes[1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'optuna_historico_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# Treinamento Definitivo

In [ ]:
model = criar_rf(best_params=best_params, seed=SEED)

t0 = time.time()
model.fit(X_train, y_train)
tempo_treino = time.time() - t0
print(f"Treinamento concluído em {tempo_treino:.2f} segundos.")

model_path = os.path.join(OUTPUT_DIR, f'{NOME_ARQUIVO}.joblib')
joblib.dump(model, model_path, compress=3)
print(f"Modelo salvo em: {model_path}")

# Curva de Convergência (F1 × Número de Árvores)

Equivalente às curvas de loss/acurácia por época das CNNs: o Random Forest não treina por
épocas, então a curva mostra o desempenho em treino e validação conforme as árvores são
adicionadas (`warm_start`). A distância entre as duas curvas indica o grau de overfitting.

In [ ]:
rf_curva = criar_rf(best_params=best_params, seed=SEED)
rf_curva.set_params(warm_start=True)

passo = 25
n_arvores, f1_treino_curva, f1_val_curva = [], [], []
for n in range(passo, best_params['n_estimators'] + 1, passo):
    rf_curva.set_params(n_estimators=n)
    rf_curva.fit(X_train, y_train)
    n_arvores.append(n)
    f1_treino_curva.append(f1_score(y_train, rf_curva.predict(X_train), average='macro'))
    f1_val_curva.append(f1_score(y_val, rf_curva.predict(X_val), average='macro'))

plt.figure(figsize=(9, 5))
plt.plot(n_arvores, f1_treino_curva, label='Treino', linewidth=2)
plt.plot(n_arvores, f1_val_curva, label='Validação', linewidth=2)
plt.title(f'Curva de Convergência — {MODELO_NOME} / {DATASET_ATIVO}', fontsize=14)
plt.xlabel('Número de árvores')
plt.ylabel('F1-Macro')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'curvas_aprendizado_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

del rf_curva
gc.collect()

# Avaliação Definitiva no Conjunto de Teste

In [ ]:
print("Avaliando o modelo final no conjunto de TESTE (dados nunca vistos)...")
y_true_test       = y_test
y_pred_probs_test = model.predict_proba(X_test)
y_pred_test       = np.argmax(y_pred_probs_test, axis=1)
test_acc          = accuracy_score(y_true_test, y_pred_test)

print(f"\nAcurácia no Teste: {test_acc:.4f}")
print(f"F1-Macro no Teste: {f1_score(y_true_test, y_pred_test, average='macro'):.4f}")

# Métricas ponderadas (as mesmas reportadas na IC, seção 8) -- mantidas para comparação
# direta com o resultado da IC; a tabela comparativa do TCC usa as médias MACRO.
print("\n--- Métricas Gerais (ponderadas, como na IC) ---")
print(f"Precisão (ponderada): {precision_score(y_true_test, y_pred_test, average='weighted', zero_division=0):.4f}")
print(f"Recall   (ponderado): {recall_score(y_true_test, y_pred_test, average='weighted', zero_division=0):.4f}")
print(f"F1-Score (ponderado): {f1_score(y_true_test, y_pred_test, average='weighted'):.4f}")

# Matriz de confusão e Avaliação Detalhada

In [ ]:
cm = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_geral, yticklabels=class_names_geral,
            linewidths=.5, cbar_kws={"shrink": .75})

plt.title(f'Matriz de Confusão — {MODELO_NOME} / {DATASET_ATIVO} (Teste)', pad=20, fontsize=16)
plt.ylabel('Classe Verdadeira (Real)', fontsize=12, labelpad=15)
plt.xlabel('Classe Predita (Modelo)', fontsize=12, labelpad=15)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'matriz_confusao_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print("\nRelatório de Classificação Detalhado (Teste):\n")
print(classification_report(y_true_test, y_pred_test, target_names=class_names_geral, zero_division=0))

# Curvas ROC, Precision-Recall e F1 por Classe

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

n_classes = len(class_names_geral)
y_true_bin = label_binarize(y_true_test, classes=list(range(n_classes)))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, class_name in enumerate(class_names_geral):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs_test[:, i])
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, linewidth=2, label=f'{class_name} (AUC={roc_auc:.2f})')

    precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_probs_test[:, i])
    ap = average_precision_score(y_true_bin[:, i], y_pred_probs_test[:, i])
    axes[1].plot(recall, precision, linewidth=2, label=f'{class_name} (AP={ap:.2f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Aleatório')
axes[0].set_title('Curvas ROC por Classe (One-vs-Rest)', fontsize=13)
axes[0].set_xlabel('Taxa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (TPR)')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Curvas Precision-Recall por Classe', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='lower left', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'curvas_roc_pr_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

f1_por_classe = f1_score(y_true_test, y_pred_test, average=None)
f1_macro = f1_score(y_true_test, y_pred_test, average='macro')

plt.figure(figsize=(10, 5))
cores = plt.cm.Blues(np.linspace(0.4, 0.85, n_classes))
bars = plt.bar(class_names_geral, f1_por_classe, color=cores, edgecolor='white', linewidth=0.8)
plt.axhline(y=f1_macro, color='red', linestyle='--', linewidth=1.8, label=f'F1-Macro = {f1_macro:.3f}')

for bar, val in zip(bars, f1_por_classe):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
             f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('F1-Score por Classe no Conjunto de Teste', fontsize=14)
plt.xlabel('Classe', fontsize=12)
plt.ylabel('F1-Score', fontsize=12)
plt.ylim(0, 1.12)
plt.legend(fontsize=11)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'f1_por_classe_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# Calibração de Confiança (ECE + Reliability Diagram)

In [ ]:
# Mesma métrica das CNNs/ViT. Atenção à interpretação: a "probabilidade" do Random Forest é a
# fração de árvores que votou na classe, não uma softmax -- florestas costumam ser
# UNDERconfident (barras ACIMA da diagonal), o oposto do típico em redes profundas.
def calcular_calibracao(y_true, y_pred, y_pred_probs, n_bins=10):
    confidences = y_pred_probs[np.arange(len(y_pred)), y_pred]
    corretos = (y_pred == y_true).astype(float)

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.clip(np.digitize(confidences, bin_edges[1:-1]), 0, n_bins - 1)

    registros = []
    ece = 0.0
    n_total = len(confidences)
    for b in range(n_bins):
        mask = bin_ids == b
        n_bin = int(mask.sum())
        if n_bin == 0:
            continue
        conf_media = float(confidences[mask].mean())
        acc_media = float(corretos[mask].mean())
        peso = n_bin / n_total
        ece += peso * abs(acc_media - conf_media)
        registros.append({
            'faixa_confianca': f'{bin_edges[b]:.1f}-{bin_edges[b + 1]:.1f}',
            'confianca_media': conf_media,
            'acuracia_media':  acc_media,
            'n_amostras':      n_bin,
        })

    return pd.DataFrame(registros), ece


df_calibracao, ece_valor = calcular_calibracao(y_true_test, y_pred_test, y_pred_probs_test)
print("── Calibração de Confiança (Teste) ──")
print(df_calibracao.to_string(index=False))
print(f"Expected Calibration Error (ECE): {ece_valor:.4f}  (quanto menor, melhor — 0 é calibração perfeita)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibração perfeita')
axes[0].bar(df_calibracao['confianca_media'], df_calibracao['acuracia_media'],
            width=0.08, alpha=0.75, edgecolor='black', color='steelblue', label='Modelo')
axes[0].set_xlabel('Confiança média (probabilidade prevista)')
axes[0].set_ylabel('Acurácia real (fração de acertos)')
axes[0].set_title(f'Reliability Diagram — {MODELO_NOME} — ECE={ece_valor:.4f}', fontsize=12)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

confidences_all = y_pred_probs_test[np.arange(len(y_pred_test)), y_pred_test]
axes[1].hist(confidences_all, bins=20, color='steelblue', edgecolor='black', alpha=0.75)
axes[1].set_xlabel('Confiança prevista (probabilidade da classe predita)')
axes[1].set_ylabel('Número de amostras')
axes[1].set_title('Distribuição de Confiança no Teste', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'calibracao_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# Correção da Calibração — Platt Scaling (sigmoid)

In [ ]:
# Temperature Scaling (usado nas CNNs/ViT) precisa de LOGITS, que o Random Forest não tem.
# O equivalente para florestas é o Platt Scaling (sigmoid por classe, one-vs-rest) ajustado
# sobre as probabilidades do modelo JÁ TREINADO (cv='prefit'), usando o conjunto de
# VALIDAÇÃO -- nunca o teste, pelo mesmo motivo de não vazar informação.
# Diferente do Temperature Scaling, a recalibração por classe PODE mudar o argmax de algumas
# amostras -- por isso a acurácia antes/depois é reportada junto com o ECE.
from sklearn.calibration import CalibratedClassifierCV

calibrador = CalibratedClassifierCV(model, method='sigmoid', cv='prefit')
calibrador.fit(X_val, y_val)

y_pred_probs_calibrado = calibrador.predict_proba(X_test)
y_pred_test_cal = np.argmax(y_pred_probs_calibrado, axis=1)

df_calibracao_pos, ece_pos = calcular_calibracao(y_true_test, y_pred_test_cal, y_pred_probs_calibrado)
print("── Calibração APÓS Platt Scaling (Teste) ──")
print(df_calibracao_pos.to_string(index=False))
print(f"ECE antes: {ece_valor:.4f}  |  ECE depois: {ece_pos:.4f}")
print(f"Acurácia antes: {test_acc:.4f}  |  Acurácia depois: {accuracy_score(y_true_test, y_pred_test_cal):.4f}")

joblib.dump(calibrador, os.path.join(OUTPUT_DIR, f'{NOME_ARQUIVO}_calibrado.joblib'), compress=3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_plot, ece_plot, titulo in [
    (axes[0], df_calibracao, ece_valor, 'Antes (sem calibração)'),
    (axes[1], df_calibracao_pos, ece_pos, 'Depois (Platt Scaling)'),
]:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Calibração perfeita')
    ax.bar(df_plot['confianca_media'], df_plot['acuracia_media'],
           width=0.08, alpha=0.75, edgecolor='black', color='steelblue', label='Modelo')
    ax.set_xlabel('Confiança média')
    ax.set_ylabel('Acurácia real')
    ax.set_title(f'{titulo} — ECE={ece_plot:.4f}', fontsize=12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'calibracao_platt_scaling_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

# Limiar de Abstenção Baseado em Confiança (Predição Seletiva)

In [ ]:
# Mesma análise das CNNs/ViT, sobre a confiança CALIBRADA (célula anterior).
confidences_cal = y_pred_probs_calibrado[np.arange(len(y_pred_test_cal)), y_pred_test_cal]
corretos_bin = (y_pred_test_cal == y_true_test)

limiares_curva = np.arange(0.0, 1.001, 0.01)
coberturas, acuracias_mantidas = [], []
for t in limiares_curva:
    mask = confidences_cal >= t
    coberturas.append(mask.mean())
    acuracias_mantidas.append(corretos_bin[mask].mean() if mask.sum() > 0 else np.nan)

plt.figure(figsize=(8, 5))
plt.plot(coberturas, acuracias_mantidas, linewidth=2, color='steelblue')
plt.axhline(y=corretos_bin.mean(), color='gray', linestyle='--', alpha=0.6,
            label=f'Acurácia sem abstenção = {corretos_bin.mean():.3f}')
plt.xlabel('Cobertura (fração de imagens respondidas)')
plt.ylabel('Acurácia nas respostas mantidas')
plt.title(f'Curva Risco × Cobertura — {MODELO_NOME} (confiança calibrada)', fontsize=13)
plt.gca().invert_xaxis()
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'risco_cobertura_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

LIMIARES_CANDIDATOS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
registros_abstencao = []
for t in LIMIARES_CANDIDATOS:
    mask = confidences_cal >= t
    n_mantidas = int(mask.sum())
    n_abstidas = int(len(mask) - n_mantidas)
    acc_mantidas = float(corretos_bin[mask].mean()) if n_mantidas > 0 else None
    erros_evitados = int(((~mask) & (~corretos_bin)).sum())
    erros_que_passam = int((mask & (~corretos_bin)).sum())
    registros_abstencao.append({
        'limiar':                  t,
        'cobertura (%)':           round(100 * n_mantidas / len(mask), 1),
        'acurácia nas mantidas':   round(acc_mantidas, 4) if acc_mantidas is not None else None,
        'n_abstidas':              n_abstidas,
        'erros_evitados':          erros_evitados,
        'erros_que_ainda_passam':  erros_que_passam,
    })

df_abstencao = pd.DataFrame(registros_abstencao)
print("── Predição Seletiva — Cobertura × Acurácia por limiar de confiança ──")
print(df_abstencao.to_string(index=False))

abstencao_path = os.path.join(OUTPUT_DIR, f'abstencao_{NOME_ARQUIVO}.csv')
df_abstencao.to_csv(abstencao_path, index=False)
print(f"Resultados salvos em: {abstencao_path}")

# XAI — Importância das Características (MDI e Permutação)

Grad-CAM e Integrated Gradients não se aplicam ao Random Forest (não há mapas convolucionais
treináveis nem gradientes). As explicações equivalentes são:
- **MDI** (Mean Decrease in Impurity) — a mesma importância mostrada na IC;
- **Importância por Permutação** no teste — mais confiável que o MDI quando há características
  correlacionadas (caso aqui: média do Gaussiano, da Mediana e do Max Pooling são quase iguais);
- **SHAP (TreeExplainer)** — contribuição de cada característica por amostra e por classe;
- **LIME por superpixels** — mesma explicação visual usada nas CNNs/ViT, permitindo comparar
  as regiões da folha que cada modelo usa.

In [ ]:
from sklearn.inspection import permutation_importance

importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Calculando importância por permutação no conjunto de teste...")
perm = permutation_importance(model, X_test, y_test, scoring='f1_macro', n_repeats=10,
                              random_state=SEED, n_jobs=-1)
indices_perm = np.argsort(perm.importances_mean)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
axes[0].bar(range(len(FEATURE_NAMES)), importances[indices], align='center', color='steelblue')
axes[0].set_xticks(range(len(FEATURE_NAMES)))
axes[0].set_xticklabels(np.array(FEATURE_NAMES)[indices], rotation=90)
axes[0].set_title('Importância das Características (MDI — como na IC)', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(range(len(FEATURE_NAMES)), perm.importances_mean[indices_perm],
            yerr=perm.importances_std[indices_perm], align='center', color='darkorange', capsize=3)
axes[1].set_xticks(range(len(FEATURE_NAMES)))
axes[1].set_xticklabels(np.array(FEATURE_NAMES)[indices_perm], rotation=90)
axes[1].set_title('Importância por Permutação (queda de F1-Macro no teste)', fontsize=13)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'{MODELO_NOME} / {DATASET_ATIVO}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'importancia_features_{NOME_ARQUIVO}.png'), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

df_importancia = pd.DataFrame({
    'feature':           FEATURE_NAMES,
    'mdi':               importances,
    'permutacao_media':  perm.importances_mean,
    'permutacao_dp':     perm.importances_std,
}).sort_values('permutacao_media', ascending=False)
print(df_importancia.to_string(index=False))
df_importancia.to_csv(os.path.join(OUTPUT_DIR, f'importancia_features_{NOME_ARQUIVO}.csv'), index=False)

# XAI — SHAP (TreeExplainer)

In [ ]:
N_SHAP = 500  # amostras do teste explicadas (TreeExplainer é exato, mas cresce com nº de árvores x profundidade)

try:
    import shap

    rng = np.random.default_rng(SEED)
    idx_shap = rng.choice(len(X_test), size=min(N_SHAP, len(X_test)), replace=False)
    X_shap = X_test[idx_shap]

    explainer_shap = shap.TreeExplainer(model)
    shap_values = explainer_shap.shap_values(X_shap, check_additivity=False)
    # Versões do SHAP diferem no formato multi-classe: lista [n_classes] de (n, f) ou array
    # (n, f, n_classes). Normaliza para lista.
    if isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
        shap_values = [shap_values[:, :, c] for c in range(shap_values.shape[2])]

    plt.figure()
    shap.summary_plot(shap_values, X_shap, feature_names=FEATURE_NAMES, class_names=class_names_geral,
                      plot_type='bar', show=False)
    plt.title(f'SHAP — Importância média por classe — {MODELO_NOME}/{DATASET_ATIVO}', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'shap_resumo_{NOME_ARQUIVO}.png'), dpi=130, bbox_inches='tight')
    plt.show()
    plt.close()

    for class_idx, class_name in enumerate(class_names_geral):
        plt.figure()
        shap.summary_plot(shap_values[class_idx], X_shap, feature_names=FEATURE_NAMES, show=False)
        plt.title(f'SHAP — {class_name}', fontsize=12)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f'shap_{NOME_ARQUIVO}_{class_name}.png'), dpi=110, bbox_inches='tight')
        plt.show()
        plt.close()
except Exception as e:
    print(f"SHAP indisponível nesta sessão ({type(e).__name__}: {e}) -- seguindo sem esta seção.")

# XAI — LIME (Explicabilidade por Superpixels)

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

def predict_fn_lime(images):
    """O LIME perturba a IMAGEM (apaga superpixels); o pipeline completo do RF é
    extração de características + predict_proba, então é isso que ele precisa chamar."""
    return model.predict_proba(extrair_features_lote(images.astype(np.float32)))

# Mapeamento arquivo → predição (mesma ordem de X_test, que foi extraído com shuffle=False)
all_test_info = [
    {'path': p, 'true_label': int(t), 'pred_label': int(pr), 'confidence': float(y_pred_probs_test[i, pr])}
    for i, (p, t, pr) in enumerate(zip(test_file_paths, y_true_test, y_pred_test))
]

explainer_lime = lime_image.LimeImageExplainer(random_state=SEED)
N_LIME_SAMPLES = 1000
explicacoes_lime = {}  # reaproveitadas na avaliação de fidelidade logo abaixo


def explicar_lime(caminho, class_idx):
    chave = (caminho, class_idx)
    if chave not in explicacoes_lime:
        img_uint8 = np.clip(np.round(carregar_imagem(caminho)), 0, 255).astype(np.uint8)
        explicacoes_lime[chave] = (img_uint8, explainer_lime.explain_instance(
            img_uint8, predict_fn_lime, labels=(class_idx,), top_labels=None,
            hide_color=0, num_samples=N_LIME_SAMPLES, batch_size=200, random_seed=SEED,
        ))
    return explicacoes_lime[chave]


print("Gerando explicações LIME por classe (pode demorar alguns minutos)...")
for class_idx, class_name in enumerate(class_names_geral):
    acertos = [d for d in all_test_info if d['true_label'] == class_idx and d['pred_label'] == class_idx]
    if not acertos:
        print(f"  [{class_name}] Sem acertos — pulando.")
        continue

    img_rgb, explanation = explicar_lime(acertos[0]['path'], class_idx)

    temp_pos, mask_pos = explanation.get_image_and_mask(class_idx, positive_only=True, num_features=6, hide_rest=False)
    temp_all, mask_all = explanation.get_image_and_mask(class_idx, positive_only=False, num_features=10, hide_rest=False)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].imshow(img_rgb)
    axes[0].set_title(f'Original\n{class_name}', fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(mark_boundaries(temp_pos.astype(np.float32) / 255, mask_pos))
    axes[1].set_title('LIME — Regiões positivas\n(verde = favor da classe)', fontsize=10)
    axes[1].axis('off')

    axes[2].imshow(mark_boundaries(temp_all.astype(np.float32) / 255, mask_all))
    axes[2].set_title('LIME — Positivo + Negativo\n(verde = favor | vermelho = contra)', fontsize=10)
    axes[2].axis('off')

    plt.suptitle(f'Explicação LIME — {MODELO_NOME}/{DATASET_ATIVO} — {class_name}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'lime_{NOME_ARQUIVO}_{class_name}.png'), dpi=120, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  [{class_name}] OK")

# Avaliação Quantitativa do LIME (Insertion/Deletion AUC)

Mesma métrica de fidelidade usada no Grad-CAM (CNNs) e no Attention Rollout (ViT) — Petsiuk
et al. (2018) —, aplicada aqui aos superpixels do LIME (ordenados do mais para o menos
importante para a classe predita).

In [ ]:
import cv2

N_FIDELIDADE_POR_CLASSE = 3
BLUR_KSIZE = 31

_trapz = getattr(np, 'trapezoid', np.trapz)


def curva_insercao_delecao_superpixels(img_rgb, explanation, class_idx, modo, blur_ksize=BLUR_KSIZE):
    """Revela ('insertion') ou apaga ('deletion') superpixels em ordem de importância do LIME
    e mede a probabilidade da classe a cada passo. Insertion: AUC alta é melhor.
    Deletion: AUC baixa é melhor."""
    segmentos = explanation.segments
    pesos = dict(explanation.local_exp[class_idx])
    ordem = sorted(np.unique(segmentos), key=lambda s: pesos.get(s, 0.0), reverse=True)

    blur = cv2.GaussianBlur(img_rgb, (blur_ksize, blur_ksize), 0).astype(np.float32)
    original = img_rgb.astype(np.float32)
    base  = blur if modo == 'insertion' else original
    fonte = original if modo == 'insertion' else blur

    frames = [base.copy()]
    atual = base.copy()
    for seg in ordem:
        atual = atual.copy()
        mask = segmentos == seg
        atual[mask] = fonte[mask]
        frames.append(atual)

    probs = model.predict_proba(extrair_features_lote(np.stack(frames, axis=0)))[:, class_idx]
    fracoes = np.linspace(0, 1, len(probs))
    return fracoes, probs, float(_trapz(probs, fracoes))


print("Avaliando fidelidade do LIME via Insertion/Deletion AUC (pode demorar alguns minutos)...")
registros_fidelidade = []
for class_idx, class_name in enumerate(class_names_geral):
    acertos = [d for d in all_test_info if d['true_label'] == class_idx and d['pred_label'] == class_idx]
    for d in acertos[:N_FIDELIDADE_POR_CLASSE]:
        img_rgb, explanation = explicar_lime(d['path'], class_idx)
        _, _, auc_ins = curva_insercao_delecao_superpixels(img_rgb, explanation, class_idx, modo='insertion')
        _, _, auc_del = curva_insercao_delecao_superpixels(img_rgb, explanation, class_idx, modo='deletion')
        registros_fidelidade.append({
            'classe':           class_name,
            'arquivo':          pathlib.Path(d['path']).name,
            'insertion_auc':    auc_ins,
            'deletion_auc':     auc_del,
            'score_fidelidade': auc_ins - auc_del,
        })

df_fidelidade = pd.DataFrame(registros_fidelidade)
print("\n── Fidelidade do LIME por amostra ──")
print(df_fidelidade.to_string(index=False))

if len(df_fidelidade):
    print("\n── Fidelidade média por classe ──")
    print(df_fidelidade.groupby('classe')[['insertion_auc', 'deletion_auc', 'score_fidelidade']].mean().to_string())
    print(f"\n── Fidelidade média geral ({len(df_fidelidade)} amostras) ──")
    print(f"  Insertion AUC  : {df_fidelidade['insertion_auc'].mean():.4f} (quanto maior, melhor)")
    print(f"  Deletion AUC   : {df_fidelidade['deletion_auc'].mean():.4f} (quanto menor, melhor)")
    print(f"  Score (Ins-Del): {df_fidelidade['score_fidelidade'].mean():.4f}")

fidelidade_path = os.path.join(OUTPUT_DIR, f'lime_fidelidade_{NOME_ARQUIVO}.csv')
df_fidelidade.to_csv(fidelidade_path, index=False)
print(f"\nResultados salvos em: {fidelidade_path}")

# Latência de Inferência (Benchmark Mobile/Offline)

In [ ]:
# Mede o pipeline COMPLETO de uma imagem (extração de características + predição), como na
# IC -- é o que o app executaria. n_jobs=1: para UMA imagem, despachar a predição em vários
# núcleos só adiciona overhead (e um celular não roda o joblib do sklearn).
N_WARMUP = 10
N_RUNS   = 100


def medir_latencia(prever_fn, img, n_warmup=N_WARMUP, n_runs=N_RUNS):
    """Retorna (média, dp, p95) em ms de extract_features + prever_fn para UMA imagem."""
    for _ in range(n_warmup):
        prever_fn(np.array([extract_features(img)], dtype=np.float32))
    tempos = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        prever_fn(np.array([extract_features(img)], dtype=np.float32))
        tempos.append((time.perf_counter() - t0) * 1000)
    return np.mean(tempos), np.std(tempos), np.percentile(tempos, 95)


sample_img = carregar_imagem(test_file_paths[0])

# Decomposição: quanto do tempo é extração e quanto é a floresta
t_ext = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    feats_sample = np.array([extract_features(sample_img)], dtype=np.float32)
    t_ext.append((time.perf_counter() - t0) * 1000)

model.set_params(n_jobs=1)
latencia_media, latencia_std, latencia_p95 = medir_latencia(model.predict_proba, sample_img)
model.set_params(n_jobs=-1)

print("── Benchmark de Inferência (scikit-learn, CPU, 1 núcleo) ──")
print(f"  Extração de features : {np.mean(t_ext):.2f} ms")
print(f"  Total (extr. + pred.): {latencia_media:.2f} ms  (dp {latencia_std:.2f} | p95 {latencia_p95:.2f})")
print(f"  FPS equivalente      : {1000 / latencia_media:.1f} imagens/s")

# Exportação do Modelo (ONNX para Flutter)

In [ ]:
def exportar_modelo(clf, X_exemplo):
    """Exporta o RF para ONNX (formato embarcável no Flutter via onnxruntime). Se
    skl2onnx/onnxruntime não estiverem disponíveis, cai para joblib (sklearn).
    Retorna (caminho, formato, prever_fn, tamanho_mb)."""
    try:
        from skl2onnx import to_onnx
        import onnxruntime as ort

        onx = to_onnx(clf, X_exemplo[:1].astype(np.float32),
                      options={RandomForestClassifier: {'zipmap': False}})
        caminho = os.path.join(OUTPUT_DIR, f'{NOME_ARQUIVO}.onnx')
        with open(caminho, 'wb') as f:
            f.write(onx.SerializeToString())

        opcoes = ort.SessionOptions()
        opcoes.intra_op_num_threads = 1
        sess = ort.InferenceSession(caminho, sess_options=opcoes, providers=['CPUExecutionProvider'])
        nome_entrada = sess.get_inputs()[0].name
        nome_probs = [o.name for o in sess.get_outputs() if 'prob' in o.name.lower()][0]

        def prever_fn(x):
            return sess.run([nome_probs], {nome_entrada: x.astype(np.float32)})[0]

        formato = 'ONNX (float32)'
    except Exception as e:
        print(f"ONNX indisponível ({type(e).__name__}: {str(e)[:300]}) -- exportando em joblib.")
        caminho = os.path.join(OUTPUT_DIR, f'{NOME_ARQUIVO}.joblib')
        if not os.path.exists(caminho):
            joblib.dump(clf, caminho, compress=3)
        recarregado = joblib.load(caminho)
        recarregado.set_params(n_jobs=1)
        prever_fn = recarregado.predict_proba
        formato = 'joblib (sklearn)'

    tamanho_mb = os.path.getsize(caminho) / (1024 ** 2)
    return caminho, formato, prever_fn, tamanho_mb


export_path, formato_exportado, prever_exportado, export_size_mb = exportar_modelo(model, X_test)
print(f"Modelo exportado: {export_path} [{formato_exportado}] — {export_size_mb:.2f} MB")

# O ONNX usa limiares float32 nas árvores -- confere se as predições batem com o sklearn
pred_exportado = np.argmax(prever_exportado(X_test), axis=1)
print(f"Concordância exportado × sklearn no teste: {np.mean(pred_exportado == y_pred_test) * 100:.2f}%")
print(f"Tamanho do .joblib (comprimido): {os.path.getsize(model_path) / (1024 ** 2):.2f} MB")
print("Obs.: no app, extract_features também precisa ser reimplementado em Dart (filtros "
      "Gaussiano/Sobel/Laplaciano/Mediana/Max Pooling) antes de chamar o modelo exportado.")

# Latência (Modelo Exportado)

In [ ]:
lat_export_media, lat_export_std, lat_export_p95 = medir_latencia(prever_exportado, sample_img)

print(f"── Benchmark de Inferência ({formato_exportado}, CPU, 1 thread) ──")
print(f"  Latência média  : {lat_export_media:.2f} ms")
print(f"  Percentil 95    : {lat_export_p95:.2f} ms")
print(f"  FPS equivalente : {1000 / lat_export_media:.1f} imagens/s")
print(f"\n  Speedup exportado vs sklearn: {latencia_media / lat_export_media:.2f}x")

# Validação Multi-Seed (Rigor Estatístico)

In [ ]:
# Mesmas seeds das CNNs/ViT -- o teste de significância pareia os experimentos seed a seed.
# Para o RF, a seed controla o bootstrap e a escolha aleatória de features em cada nó.
SEEDS_ADICIONAIS = [123, 2024]


def metricas_teste(y_true, y_pred, seed):
    return {
        'seed':            seed,
        'accuracy':        accuracy_score(y_true, y_pred),
        'f1_macro':        f1_score(y_true, y_pred, average='macro'),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':    recall_score(y_true, y_pred, average='macro', zero_division=0),
    }


def treinar_com_seed(seed, best_params):
    """Treina o RF final com uma seed distinta e retorna as métricas no teste."""
    m = criar_rf(best_params=best_params, seed=seed)
    m.fit(X_train, y_train)
    metricas = metricas_teste(y_test, m.predict(X_test), seed)
    del m
    gc.collect()
    return metricas


def salvar_multiseed(resultados):
    df = pd.DataFrame(resultados)
    cols = ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']
    media, desvio = df[cols].mean(), df[cols].std()

    df_export = df.copy()
    df_export.insert(0, 'dataset', DATASET_ATIVO)
    df_export.insert(0, 'modelo', MODELO_NOME)
    caminho = os.path.join(OUTPUT_DIR, f'multiseed_{NOME_ARQUIVO}.csv')
    df_export.to_csv(caminho, index=False)
    print(f"Resultados por seed salvos em: {caminho}")
    return df, media, desvio


# Reaproveita o modelo canônico (SEED=42) já avaliado acima -- não retreina
resultados_multiseed = [metricas_teste(y_true_test, y_pred_test, SEED)]
for s in SEEDS_ADICIONAIS:
    print(f"Treinando com seed={s} ({MODELO_NOME} / {DATASET_ATIVO})...")
    resultados_multiseed.append(treinar_com_seed(s, best_params))
    print(f"  Concluído. Acurácia={resultados_multiseed[-1]['accuracy']:.4f} | F1-Macro={resultados_multiseed[-1]['f1_macro']:.4f}")

df_multiseed, media_multiseed, desvio_multiseed = salvar_multiseed(resultados_multiseed)
print("\n── Resultados por seed ──")
print(df_multiseed.to_string(index=False))
print(f"\n── Média ± Desvio-padrão ({len(df_multiseed)} seeds) ──")
for col in ['accuracy', 'f1_macro', 'precision_macro', 'recall_macro']:
    print(f"  {col:17s}: {media_multiseed[col]:.4f} ± {desvio_multiseed[col]:.4f}")

# Linha-Resumo para a Tabela Comparativa (Modelos × Datasets)

In [ ]:
# MESMOS nomes de coluna do resultados_comparativos do mobilenet.ipynb e do transformers.ipynb
# -- a Tabela Comparativa de qualquer um dos notebooks concatena as três famílias de modelos.
def registrar_linha_comparativa(media, desvio, n_seeds, lat_memoria, lat_exportado, tamanho_mb, formato):
    linha = {
        'Modelo':                    MODELO_NOME,
        'Dataset':                   DATASET_ATIVO,
        'Acurácia (méd)':            round(media['accuracy'], 4),
        'Acurácia (dp)':             round(desvio['accuracy'], 4),
        'F1-Macro (méd)':            round(media['f1_macro'], 4),
        'F1-Macro (dp)':             round(desvio['f1_macro'], 4),
        'Precision-Macro (méd)':     round(media['precision_macro'], 4),
        'Recall-Macro (méd)':        round(media['recall_macro'], 4),
        'N_seeds':                   n_seeds,
        'Lat. Em Memória (ms)':      round(lat_memoria, 2),
        'Lat. Exportado (ms)':       round(lat_exportado, 2),
        'Tamanho Exportado (MB)':    round(tamanho_mb, 2),
        'Formato Exportado':         formato,
    }
    caminho = os.path.join(OUTPUT_DIR, f'resultados_comparativos_{NOME_ARQUIVO}.csv')
    pd.DataFrame([linha]).to_csv(caminho, index=False)
    print(f"Linha salva em: {caminho}")
    return linha


linha_rf = registrar_linha_comparativa(
    media_multiseed, desvio_multiseed, len(df_multiseed),
    latencia_media, lat_export_media, export_size_mb, formato_exportado,
)
print(pd.DataFrame([linha_rf]).to_string(index=False))

# Automação da Avaliação Cruzada (Random Forest × Dataset)

In [ ]:
def executar_experimento_completo(dataset_name, n_trials=N_TRIALS_RF, seeds_adicionais=None):
    """
    Pipeline completo do RF para UM dataset, reaproveitando as funções acima
    (carregar_features, buscar_hiperparametros, criar_rf, treinar_com_seed, salvar_multiseed,
    medir_latencia, exportar_modelo, registrar_linha_comparativa).

    Mesmo escopo das automações do mobilenet.ipynb e do transformers.ipynb: busca + treino +
    teste + multi-seed + latência + exportação + linha comparativa (sem XAI/calibração). Ao
    final, as globais ficam apontando para este dataset -- as células manuais de matriz de
    confusão, calibração e XAI podem ser re-executadas sobre o último experimento.
    """
    global X_train, y_train, X_val, y_val, X_test, y_test, test_file_paths, class_names_geral
    global study, best_params, model, model_path, tempo_treino
    global y_true_test, y_pred_probs_test, y_pred_test, test_acc
    global resultados_multiseed, df_multiseed, media_multiseed, desvio_multiseed
    global sample_img, latencia_media, latencia_std, latencia_p95
    global export_path, formato_exportado, prever_exportado, export_size_mb
    global lat_export_media, lat_export_std, lat_export_p95

    if seeds_adicionais is None:
        seeds_adicionais = SEEDS_ADICIONAIS

    # ── 1. Ativa o dataset e carrega as features ──────────────────────────────
    ativar_dataset(dataset_name)
    print(f"\n{'=' * 80}\n  EXPERIMENTO: {MODELO_NOME} × {DATASET_ATIVO}\n{'=' * 80}")
    X_train, y_train, X_val, y_val, X_test, y_test, test_file_paths, class_names_geral = carregar_features(dataset_name)
    print(f"Classes: {class_names_geral} | Treino: {X_train.shape} | Val: {X_val.shape} | Teste: {X_test.shape}")

    # ── 2. Busca de hiperparâmetros (retomável via SQLite) ────────────────────
    study, best_params = buscar_hiperparametros(n_trials)
    print(f"Melhor F1-Macro (val): {study.best_value:.4f} | Parâmetros: {best_params}")

    # ── 3. Treino definitivo + teste ──────────────────────────────────────────
    model = criar_rf(best_params=best_params, seed=SEED)
    t0 = time.time()
    model.fit(X_train, y_train)
    tempo_treino = time.time() - t0
    model_path = os.path.join(OUTPUT_DIR, f'{NOME_ARQUIVO}.joblib')
    joblib.dump(model, model_path, compress=3)

    y_true_test = y_test
    y_pred_probs_test = model.predict_proba(X_test)
    y_pred_test = np.argmax(y_pred_probs_test, axis=1)
    test_acc = accuracy_score(y_true_test, y_pred_test)
    print(f"Teste — Acurácia: {test_acc:.4f} | F1-Macro: {f1_score(y_true_test, y_pred_test, average='macro'):.4f} "
          f"| Treino em {tempo_treino:.1f}s")

    # ── 4. Multi-seed ─────────────────────────────────────────────────────────
    resultados_multiseed = [metricas_teste(y_true_test, y_pred_test, SEED)]
    for s in seeds_adicionais:
        resultados_multiseed.append(treinar_com_seed(s, best_params))
        print(f"  seed={s} -> Acurácia={resultados_multiseed[-1]['accuracy']:.4f} | F1-Macro={resultados_multiseed[-1]['f1_macro']:.4f}")
    df_multiseed, media_multiseed, desvio_multiseed = salvar_multiseed(resultados_multiseed)

    # ── 5. Latência + exportação ──────────────────────────────────────────────
    sample_img = carregar_imagem(test_file_paths[0])
    model.set_params(n_jobs=1)
    latencia_media, latencia_std, latencia_p95 = medir_latencia(model.predict_proba, sample_img)
    model.set_params(n_jobs=-1)
    export_path, formato_exportado, prever_exportado, export_size_mb = exportar_modelo(model, X_test)
    lat_export_media, lat_export_std, lat_export_p95 = medir_latencia(prever_exportado, sample_img)
    print(f"Latência — sklearn: {latencia_media:.2f} ms | {formato_exportado}: {lat_export_media:.2f} ms "
          f"| Tamanho: {export_size_mb:.2f} MB")

    # ── 6. Linha comparativa (grava CSV por dataset em disco) ─────────────────
    linha = registrar_linha_comparativa(
        media_multiseed, desvio_multiseed, len(df_multiseed),
        latencia_media, lat_export_media, export_size_mb, formato_exportado,
    )
    gc.collect()
    print(f"Concluído: {NOME_ARQUIVO}.")
    return linha

In [ ]:
# ── Executa a matriz Random Forest × Dataset automaticamente ─────────────────────
# Padrão True (como no transformers.ipynb): o RF não tem o vazamento de memória nativa do
# TensorFlow que motivou a guarda de "Run All" no mobilenet.ipynb. E não há retrabalho num
# "Run All": as células manuais acima já gravaram outputs/RandomForest_{DATASET_ATIVO}/
# resultados_comparativos_*.csv, então combinacoes_ja_feitas() pula o dataset ativo (com
# XAI completo) e a automação só roda os demais.
RODAR_AVALIACAO_CRUZADA = True

DATASETS_PARA_RODAR = list(DATASETS.keys())   # ex.: ['jmuben'] para rodar só um dataset

N_TRIALS_BATCH = N_TRIALS_RF

# Para retomar de uma sessão anterior do Kaggle, acrescente o path do Dataset de saída que
# você copiou de volta como input, ex.: '/kaggle/input/meu-outputs-anteriores/outputs'.
CAMINHOS_OUTPUTS_ANTERIORES = ['outputs']


def combinacoes_ja_feitas(caminhos):
    feitas = set()
    for base in caminhos:
        arquivos = glob.glob(os.path.join(base, 'resultados_comparativos.csv'))
        arquivos += glob.glob(os.path.join(base, '*', 'resultados_comparativos_*.csv'))
        for arq in arquivos:
            try:
                for _, row in pd.read_csv(arq)[['Modelo', 'Dataset']].iterrows():
                    feitas.add((row['Modelo'], row['Dataset']))
            except Exception as e:
                print(f"Aviso: não foi possível ler {arq} ({e})")
    return feitas


if not RODAR_AVALIACAO_CRUZADA:
    print('RODAR_AVALIACAO_CRUZADA=False -- pulando a automação. Mude para True nesta célula '
          'para executar a matriz Random Forest × Dataset.')
else:
    ja_feitas = combinacoes_ja_feitas(CAMINHOS_OUTPUTS_ANTERIORES)
    datasets_a_rodar = [d for d in DATASETS_PARA_RODAR if (MODELO_NOME, d) not in ja_feitas]
    pulados = [d for d in DATASETS_PARA_RODAR if (MODELO_NOME, d) in ja_feitas]

    if pulados:
        print(f"Pulando {len(pulados)} dataset(s) já concluído(s) na planilha: {pulados}")
    print(f"Avaliação cruzada ({MODELO_NOME}): {len(datasets_a_rodar)} dataset(s) a treinar — {datasets_a_rodar}")

    for dataset_name in datasets_a_rodar:
        executar_experimento_completo(dataset_name, n_trials=N_TRIALS_BATCH)

# Importar Resultados das CNNs e do ViT (para a Tabela Comparativa e o Teste de Significância)

O teste de significância compara modelos **no mesmo dataset**, pareando as seeds (42, 123, 2024 —
as mesmas nos três notebooks). Para isso, este notebook precisa dos CSVs que o `mobilenet.ipynb`
e o `transformers.ipynb` gravam em `outputs/{Modelo}_{dataset}/`:

| Arquivo | Usado em |
|---|---|
| `multiseed_{Modelo}_{dataset}.csv` | Teste de significância (F1-Macro por seed) |
| `resultados_comparativos_{Modelo}_{dataset}.csv` | Tabela comparativa |

**Como anexar no Kaggle:**
1. No notebook das CNNs (e no do ViT), depois de treinar, clique em **Save Version → Save & Run All**
   (ou "Quick Save" se já executou tudo) — o conteúdo de `/kaggle/working/outputs` vira o *Output* da versão.
2. Neste notebook: **Add Input → aba "Notebooks"** → selecione o notebook das CNNs (repita para o ViT).
   Ele aparece em `/kaggle/input/<nome-do-notebook>/`.
3. Rode a célula abaixo: ela procura esses CSVs em `/kaggle/input` e copia para `outputs/`
   (sem sobrescrever os do Random Forest). Depois rode as células de Tabela Comparativa e
   Teste de Significância.

Alternativa: baixe a pasta `outputs/` dos outros notebooks, suba como um Kaggle Dataset e anexe aqui
— a célula também encontra os arquivos dessa forma.

In [ ]:
import shutil

# Onde procurar -- acrescente outro caminho se os CSVs estiverem fora de /kaggle/input
BASES_EXTERNAS = ['/kaggle/input']
PADROES_EXTERNOS = ['multiseed_*.csv', 'resultados_comparativos_*.csv']

copiados = []
for base in BASES_EXTERNAS:
    for padrao in PADROES_EXTERNOS:
        for origem in glob.glob(os.path.join(base, '**', 'outputs', '*', padrao), recursive=True):
            pasta = os.path.basename(os.path.dirname(origem))   # ex.: MobileNetV2_bracol
            if pasta.startswith(f'{MODELO_NOME}_'):
                continue   # os resultados do RF desta sessão têm prioridade
            destino = os.path.join('outputs', pasta, os.path.basename(origem))
            if os.path.exists(destino):
                continue
            os.makedirs(os.path.dirname(destino), exist_ok=True)
            shutil.copy2(origem, destino)
            copiados.append(destino)

print(f"{len(copiados)} arquivo(s) importado(s):")
for c in copiados:
    print(f"  {c}")

# ── Diagnóstico: quais pares Modelo × Dataset já podem ser comparados ─────────
disponiveis = sorted(glob.glob(os.path.join('outputs', '*', 'multiseed_*.csv')))
if disponiveis:
    df_disp = pd.concat([pd.read_csv(f)[['modelo', 'dataset']].drop_duplicates() for f in disponiveis])
    resumo = df_disp.groupby('dataset')['modelo'].apply(sorted)
    print("\nModelos com resultados multi-seed, por dataset:")
    for ds_nome, modelos in resumo.items():
        status = 'OK para comparar' if len(modelos) >= 2 else 'só 1 modelo -- falta anexar os outros'
        print(f"  {ds_nome:22s}: {modelos}  -> {status}")
if not copiados:
    print("\nNenhum CSV externo encontrado. Confira se o notebook das CNNs/ViT foi anexado via "
          "Add Input e se a versão salva contém a pasta outputs/ (rode `!find /kaggle/input -name "
          "'multiseed_*.csv'` para conferir).")

# Tabela Comparativa — Modelos × Datasets

In [ ]:
# Junta TODAS as linhas-resumo resultados_comparativos_*.csv -- deste notebook
# (outputs/RandomForest_*) e dos notebooks das CNNs (mobilenet.ipynb) e do ViT
# (transformers.ipynb).
#
# COMO TRAZER OS RESULTADOS DOS OUTROS NOTEBOOKS (Kaggle): no painel lateral, "+ Add Input"
# → aba "Notebooks" → selecione o notebook das CNNs e o do ViT (precisam ter sido salvos com
# "Save Version" + "Save output"). As pastas outputs/ deles aparecem dentro de /kaggle/input/
# e são encontradas automaticamente pela busca recursiva abaixo. Alternativa: suba as pastas
# outputs/ como um Dataset do Kaggle e anexe-o como Input -- funciona igual.
CAMINHOS_RESULTADOS = ['outputs', '/kaggle/input']


def buscar_resultados(padrao):
    """Procura `padrao` recursivamente em CAMINHOS_RESULTADOS. Se o mesmo nome de arquivo
    aparecer em mais de um lugar, fica a 1a ocorrência -- outputs/ desta sessão tem
    prioridade sobre cópias antigas anexadas em /kaggle/input."""
    encontrados = {}
    for base in CAMINHOS_RESULTADOS:
        for arq in sorted(glob.glob(os.path.join(base, '**', padrao), recursive=True)):
            encontrados.setdefault(os.path.basename(arq), arq)
    return sorted(encontrados.values())


arquivos_comparativos = buscar_resultados('resultados_comparativos_*.csv')

if arquivos_comparativos:
    df_comp = pd.concat([pd.read_csv(f) for f in arquivos_comparativos], ignore_index=True)
    df_comp = df_comp.drop_duplicates(subset=['Modelo', 'Dataset'], keep='first')
    print(f"{len(arquivos_comparativos)} arquivo(s) de resultados encontrado(s): "
          f"{sorted(df_comp['Modelo'].unique())}")
    df_comp = df_comp.sort_values(['Dataset', 'F1-Macro (méd)'], ascending=[True, False]).reset_index(drop=True)

    print("=" * 75)
    print("  TABELA COMPARATIVA — Modelos × Datasets")
    print("=" * 75)
    print(df_comp.to_string(index=True))
    print("=" * 75)

    comparativos_path = os.path.join('outputs', 'resultados_comparativos.csv')
    df_comp.to_csv(comparativos_path, index=False)
    print(f"Tabela salva em: {comparativos_path}")

    # Matriz Dataset × Modelo do F1-Macro (visão da Avaliação Cruzada)
    matriz_f1 = df_comp.pivot_table(index='Modelo', columns='Dataset', values='F1-Macro (méd)')
    plt.figure(figsize=(1.8 * max(len(matriz_f1.columns), 2) + 3, 0.6 * len(matriz_f1) + 2))
    sns.heatmap(matriz_f1, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1, linewidths=.5)
    plt.title('Avaliação Cruzada — F1-Macro médio (Modelo × Dataset)', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join('outputs', 'matriz_avaliacao_cruzada_f1.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    print("Nenhum resultado registrado. Execute as células de treinamento e avaliação primeiro.")

# Teste de Significância Estatística entre Modelos (F1-Macro pareado por seed)

In [ ]:
import itertools
from scipy import stats

# Mesma lógica do mobilenet.ipynb: varre os multiseed_*.csv (valores BRUTOS por seed) e
# compara pares de modelos no MESMO dataset, pareando pelas seeds em comum. Usa a mesma busca
# da Tabela Comparativa (outputs/ + /kaggle/input) -- ver lá como anexar os resultados das
# CNNs e do ViT.
arquivos_multiseed = buscar_resultados('multiseed_*.csv')

if len(arquivos_multiseed) < 2:
    print(f"Encontrado(s) {len(arquivos_multiseed)} arquivo(s) multiseed em outputs/*/multiseed_*.csv.")
    print("São necessárias pelo menos 2 combinações Modelo × Dataset no mesmo dataset para comparar "
          "estatisticamente. Anexe a saída dos notebooks das CNNs/ViT como Input (ver célula "
          "da Tabela Comparativa).")
else:
    df_todos = pd.concat([pd.read_csv(f) for f in arquivos_multiseed], ignore_index=True)
    df_todos = df_todos.drop_duplicates(subset=['modelo', 'dataset', 'seed'], keep='first')
    df_todos['experimento'] = df_todos['modelo'] + ' / ' + df_todos['dataset']

    grupos_por_dataset = df_todos.groupby('dataset')['experimento'].unique()

    registros_testes = []
    for dataset_nome, experimentos in grupos_por_dataset.items():
        if len(experimentos) < 2:
            continue
        for exp_a, exp_b in itertools.combinations(sorted(experimentos), 2):
            serie_a = df_todos[df_todos['experimento'] == exp_a].set_index('seed')['f1_macro']
            serie_b = df_todos[df_todos['experimento'] == exp_b].set_index('seed')['f1_macro']

            seeds_comuns = sorted(set(serie_a.index) & set(serie_b.index))
            if len(seeds_comuns) < 2:
                continue
            a = serie_a.loc[seeds_comuns].values
            b = serie_b.loc[seeds_comuns].values

            t_stat, p_ttest = stats.ttest_rel(a, b)

            if len(seeds_comuns) >= 6:
                try:
                    w_stat, p_wilcoxon = stats.wilcoxon(a, b)
                except ValueError:
                    p_wilcoxon = np.nan
            else:
                p_wilcoxon = np.nan

            registros_testes.append({
                'dataset':           dataset_nome,
                'modelo_a':          exp_a,
                'modelo_b':          exp_b,
                'n_seeds':           len(seeds_comuns),
                'f1_macro_a (méd)':  round(a.mean(), 4),
                'f1_macro_b (méd)':  round(b.mean(), 4),
                'diferenca_media':   round(a.mean() - b.mean(), 4),
                'p_valor_ttest':     round(p_ttest, 4),
                'significativo_5%':  'Sim' if p_ttest < 0.05 else 'Não',
                'p_valor_wilcoxon':  round(p_wilcoxon, 4) if not np.isnan(p_wilcoxon) else 'N/A (n<6)',
            })

    if registros_testes:
        df_testes = pd.DataFrame(registros_testes)
        print("=" * 100)
        print("  TESTES DE SIGNIFICÂNCIA ESTATÍSTICA — F1-Macro pareado por seed (teste t pareado)")
        print("=" * 100)
        print(df_testes.to_string(index=False))
        print("=" * 100)
        print("\nAviso: com apenas 3 seeds por experimento, o teste t pareado tem baixo poder "
              "estatístico — trate p<0.05 como indício, não como prova definitiva.")

        testes_path = os.path.join('outputs', 'testes_significancia.csv')
        df_testes.to_csv(testes_path, index=False)
        print(f"\nResultados salvos em: {testes_path}")
    else:
        print(f"Nenhum par de experimentos compartilha dataset e seeds suficientes para comparação. "
              f"Modelos encontrados: {sorted(df_todos['modelo'].unique())}. Se só aparece "
              f"'{MODELO_NOME}', anexe a saída dos notebooks das CNNs/ViT como Input (ver célula "
              f"da Tabela Comparativa) e rode esta célula de novo.")